# Insurance Claim Fraud Detection

**Tabular Classification Project**

## 2 · Project Overview

Detect **fraudulent auto insurance claims** from claim details, policy information, and incident characteristics. The dataset has ~5,000 claims with ~15% fraud rate — a moderately imbalanced classification problem common in the insurance industry.

## 3 · Learning Objectives

1. Perform EDA and target analysis on a classification dataset.
2. Handle categorical encoding, missing values, and class imbalance.
3. Build a baseline model and compare against modern boosting models.
4. Use LazyPredict for rapid classifier benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, and XGBoost with GPU auto-detection.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given claim details (amount, type), policy information (premium, deductible, coverage), and incident details (severity, witnesses, police report), predict whether the claim is fraudulent (is_fraud = 1).

## 5 · Why This Project Matters

- Insurance fraud costs the US industry **$80+ billion/year** (FBI estimate).
- Fraudulent claims raise premiums for all policyholders.
- ~15% fraud rate is more tractable than credit card fraud (~1%) but still imbalanced.
- This teaches combining structured business rules with ML models.
- Explainability is critical — investigators need reasons, not just scores.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| Rows | ~5,000 |
| Features | 14 (claim_amount, policy_premium, deductible, coverage_type, incident_severity, num_witnesses, police_report, insured_age, insured_gender, vehicle_age, prior_claims, months_as_customer, claim_day_of_week, umbrella_limit) |
| Target | `is_fraud` (binary: 0=legitimate, 1=fraud) |
| Class balance | ~85% legitimate, ~15% fraud |
| Missing values | None |

## 7 · Dataset Source and License Notes

**Source**: Synthetic auto insurance claims dataset for ML practice.
**License**: Educational / public.
**Note**: Mimics real insurance claim patterns with ~15% fraud rate.

## 8 · Environment Setup

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg.replace('-','_'))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, confusion_matrix,
                              classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "is_fraud"
SEED = 42
SAVE_DIR = os.path.dirname(os.path.abspath('__dummy__'))
print(f"Save dir: {SAVE_DIR}")

Save dir: E:\Github\Machine-Learning-Projects\Classification\Insurance Claim Fraud Detection


## 11 · Dataset Download or Loading

In [4]:
np.random.seed(SEED)
n = 5000
fraud_rate = 0.15
n_fraud = int(n * fraud_rate)
n_legit = n - n_fraud

# Claim amounts
claim_legit = np.random.lognormal(7.5, 0.8, n_legit).clip(500, 50000).round(2)
claim_fraud = np.random.lognormal(8.5, 0.9, n_fraud).clip(2000, 100000).round(2)
claim_amount = np.concatenate([claim_legit, claim_fraud])

premium = np.random.normal(1200, 400, n).clip(300, 3000).round(2)
deductible = np.random.choice([250, 500, 1000, 1500, 2000], n, p=[0.15, 0.35, 0.3, 0.1, 0.1])
coverage = np.random.choice(['basic', 'standard', 'premium'], n, p=[0.3, 0.45, 0.25])

sev_legit = np.random.choice(['minor', 'moderate', 'severe', 'total_loss'], n_legit, p=[0.4, 0.35, 0.2, 0.05])
sev_fraud = np.random.choice(['minor', 'moderate', 'severe', 'total_loss'], n_fraud, p=[0.1, 0.2, 0.4, 0.3])
severity = np.concatenate([sev_legit, sev_fraud])

wit_legit = np.random.poisson(1.5, n_legit).clip(0, 5)
wit_fraud = np.random.poisson(0.5, n_fraud).clip(0, 3)
witnesses = np.concatenate([wit_legit, wit_fraud])

police_legit = np.random.binomial(1, 0.7, n_legit)
police_fraud = np.random.binomial(1, 0.3, n_fraud)
police_report = np.concatenate([police_legit, police_fraud])

age = np.random.normal(42, 13, n).clip(18, 80).astype(int)
gender = np.random.choice(['M', 'F'], n)
vehicle_age = np.random.poisson(5, n).clip(0, 25)

prior_legit = np.random.poisson(0.8, n_legit).clip(0, 5)
prior_fraud = np.random.poisson(2.0, n_fraud).clip(0, 8)
prior_claims = np.concatenate([prior_legit, prior_fraud])

months_customer = np.random.exponential(60, n).clip(1, 300).astype(int)
claim_dow = np.random.randint(0, 7, n)
umbrella = np.random.choice([0, 1000000, 2000000, 3000000], n, p=[0.4, 0.3, 0.2, 0.1])

is_fraud = np.array([0]*n_legit + [1]*n_fraud)

df = pd.DataFrame({
    'claim_amount': claim_amount, 'policy_premium': premium, 'deductible': deductible,
    'coverage_type': coverage, 'incident_severity': severity, 'num_witnesses': witnesses,
    'police_report': police_report, 'insured_age': age, 'insured_gender': gender,
    'vehicle_age': vehicle_age, 'prior_claims': prior_claims,
    'months_as_customer': months_customer, 'claim_day_of_week': claim_dow,
    'umbrella_limit': umbrella, 'is_fraud': is_fraud,
})
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Dataset shape: {df.shape}")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
df.head()

Dataset shape: (5000, 15)
Fraud rate: 15.00%


,claim_amount,policy_premium,deductible,coverage_type,incident_severity,num_witnesses,police_report,insured_age,insured_gender,vehicle_age,prior_claims,months_as_customer,claim_day_of_week,umbrella_limit,is_fraud
0,1163.34,715.94,500,premium,moderate,2,0,43,M,8,0,33,6,0,0
1,629.30,784.31,1000,standard,minor,3,1,20,F,2,0,138,3,0,0
2,4123.53,1393.12,1000,standard,minor,0,1,39,F,3,2,3,5,3000000,0
3,1056.48,1800.11,500,premium,severe,2,1,74,F,8,2,99,6,0,0
4,2936.01,365.68,1000,premium,moderate,0,0,46,M,9,0,157,1,0,0


## 12 · Data Validation Checks

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
if df.isnull().sum().sum() == 0:
    print("No missing values.")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nTarget distribution:\n{df[TARGET].value_counts()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")

DATA VALIDATION

Shape: (5000, 15)

Missing values:
Series([], dtype: int64)
No missing values.

Duplicate rows: 0

Target distribution:
is_fraud
0    4250
1     750
Name: count, dtype: int64

Dtypes:
claim_amount          float64
policy_premium        float64
deductible              int64
coverage_type          object
incident_severity      object
num_witnesses           int32
police_report           int32
insured_age             int64
insured_gender         object
vehicle_age             int32
prior_claims            int32
months_as_customer      int64
claim_day_of_week       int32
umbrella_limit          int64
is_fraud                int64
dtype: object

Target 'is_fraud' confirmed.


## 13 · Exploratory Data Analysis

In [6]:
num_cols = df.select_dtypes(include='number').columns.drop(TARGET).tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(['claim_amount', 'policy_premium', 'num_witnesses', 'prior_claims', 'insured_age', 'vehicle_age']):
    ax = axes[i // 3, i % 3]
    df[df[TARGET]==0][col].hist(bins=25, ax=ax, alpha=0.6, label='Legit', color='steelblue')
    df[df[TARGET]==1][col].hist(bins=25, ax=ax, alpha=0.6, label='Fraud', color='coral')
    ax.set_title(col); ax.legend()
plt.suptitle("Feature Distributions by Fraud Status", fontsize=14)
plt.tight_layout()
plt.savefig('eda_numeric.png', dpi=100, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for i, col in enumerate(['incident_severity', 'coverage_type']):
    ct = pd.crosstab(df[col], df[TARGET], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=axes[i], color=['steelblue', 'coral'])
    axes[i].set_title(f"Fraud Rate by {col}")
plt.tight_layout()
plt.show()
print(f"Numeric: {len(num_cols)}, Categorical: {len(cat_cols)}")

Numeric: 11, Categorical: 3


## 14 · Target Analysis

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title("Insurance Claim Fraud Distribution")
axes[0].set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
df[TARGET].value_counts(normalize=True).plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['steelblue', 'coral'])
axes[1].set_title("Class Proportions"); axes[1].set_ylabel('')
plt.tight_layout(); plt.show()
print(f"Class distribution:\n{df[TARGET].value_counts()}")
print(f"\nImbalance ratio: {df[TARGET].value_counts().iloc[0] / max(df[TARGET].value_counts().iloc[1], 1):.1f}:1")

Class distribution:
is_fraud
0    4250
1     750
Name: count, dtype: int64

Imbalance ratio: 5.7:1


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 split to preserve class distribution.

In [8]:
cat_cols = df.select_dtypes(include='object').columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
    df[cat_cols] = oe.fit_transform(df[cat_cols])

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target dist: {y_train.value_counts().to_dict()}")

Train: (4000, 14), Test: (1000, 14)
Train target dist: {0: 3400, 1: 600}


## 16 · Preprocessing Strategy

Categorical features encoded via OrdinalEncoder. Missing values handled before split. Tree-based models handle raw features without scaling.

## 17 · Feature Engineering

In [9]:
clean = [c.replace('-', '_').replace(' ', '_').replace('.', '_') for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (14): ['claim_amount', 'policy_premium', 'deductible', 'coverage_type', 'incident_severity', 'num_witnesses', 'police_report', 'insured_age', 'insured_gender', 'vehicle_age', 'prior_claims', 'months_as_customer', 'claim_day_of_week', 'umbrella_limit']


## 18 · Baseline Model

Logistic Regression as baseline.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)
y_prob_bl = baseline.predict_proba(X_test)[:, 1] if len(np.unique(y_train)) == 2 else None

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {accuracy_score(y_test, y_pred_bl):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_bl, average='weighted'):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bl, average='weighted'):.4f}")
if y_prob_bl is not None:
    print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob_bl):.4f}")

BASELINE: Logistic Regression
Accuracy : 0.9090
Precision: 0.9047
Recall   : 0.9090
F1       : 0.9061
ROC-AUC  : 0.9405


## 19 · LazyPredict Benchmark

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                             Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision  Recall  Time Taken
Model                                                                                                      
NearestCentroid                 0.924           0.911373  0.957514  0.927851   0.936735   0.924    0.027092
AdaBoostClassifier              0.949           0.876667  0.976145  0.947721   0.947494   0.949    0.238292
XGBClassifier                   0.941           0.863725  0.969255  0.939705   0.939197   0.941    0.295728
CatBoostClassifier              0.946           0.861176  0.977553  0.944028   0.944214   0.946    3.217865
LGBMClassifier                  0.942           0.858824  0.974667  0.940266   0.939973   0.942    3.750826
PassiveAggressiveClassifier     0.926           0.857647  0.937247  0.926201   0.926415   0.926    0.020646
RandomForestClassifier          0.949           0.851961  0.973408  0.946146   0.948266   0.949    0.4

## 20 · FLAML AutoML Run

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
_flaml_metric = "macro_f1" if y_train.nunique() > 2 else "f1"
flaml_automl.fit(
    X_train, y_train,
    task="classification", time_budget=60, metric=_flaml_metric,
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best: {flaml_automl.best_estimator}")
print(f"Accuracy : {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_flaml, average='weighted'):.4f}")

FLAML best: lgbm
Accuracy : 0.9470
F1       : 0.9455


## 21 · CatBoost, LightGBM, XGBoost

GPU auto-detected with CPU fallback.

In [13]:
import gc

def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}

# CatBoost
from catboost import CatBoostClassifier
try:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            task_type="GPU", devices="0", verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
except Exception:
    cb = CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6,
                            verbose=0, random_seed=SEED)
    cb.fit(X_train, y_train)
results["CatBoost"] = cb.predict(X_test)
print(f"CatBoost  Acc: {accuracy_score(y_test, results['CatBoost']):.4f}  F1: {f1_score(y_test, results['CatBoost'], average='weighted'):.4f}")
gpu_cleanup()

# LightGBM
import lightgbm as lgbm
try:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              device="gpu", verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
except Exception:
    lg = lgbm.LGBMClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=SEED)
    lg.fit(X_train, y_train)
results["LightGBM"] = lg.predict(X_test)
print(f"LightGBM  Acc: {accuracy_score(y_test, results['LightGBM']):.4f}  F1: {f1_score(y_test, results['LightGBM'], average='weighted'):.4f}")
gpu_cleanup()

# XGBoost
from xgboost import XGBClassifier
try:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               device="cuda", tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
except Exception:
    xgb_model = XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               tree_method="hist", verbosity=0,
                               random_state=SEED, use_label_encoder=False, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)
results["XGBoost"] = xgb_model.predict(X_test)
print(f"XGBoost   Acc: {accuracy_score(y_test, results['XGBoost']):.4f}  F1: {f1_score(y_test, results['XGBoost'], average='weighted'):.4f}")
gpu_cleanup()

results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost  Acc: 0.9490  F1: 0.9474


LightGBM  Acc: 0.9370  F1: 0.9342


XGBoost   Acc: 0.9440  F1: 0.9429


## 22 · Top 3 Model Selection

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average='weighted'), 4),
        "Precision": round(precision_score(y_test, yp, average='weighted'), 4),
        "Recall": round(recall_score(y_test, yp, average='weighted'), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())
top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
CatBoost                0.949  0.9474     0.9474   0.949
FLAML                   0.947  0.9455     0.9453   0.947
XGBoost                 0.944  0.9429     0.9424   0.944
LightGBM                0.937  0.9342     0.9344   0.937
Logistic Regression     0.909  0.9061     0.9047   0.909

Top 3: ['CatBoost', 'FLAML', 'XGBoost']


## 23 · Final Training and Evaluation of Top 3

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap='Blues')
    f1 = f1_score(y_test, yp, average='weighted')
    acc = accuracy_score(y_test, yp)
    axes[i].set_title(f"{name}\nAcc={acc:.4f} F1={f1:.4f}")
plt.suptitle("Top 3 — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig('top3_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

print("\nDetailed Classification Reports — Top 3:")
for name in top3_names:
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)
    print(classification_report(y_test, results[name]))


Detailed Classification Reports — Top 3:

  CatBoost
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       850
           1       0.88      0.76      0.82       150

    accuracy                           0.95      1000
   macro avg       0.92      0.87      0.89      1000
weighted avg       0.95      0.95      0.95      1000


  FLAML
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       850
           1       0.87      0.76      0.81       150

    accuracy                           0.95      1000
   macro avg       0.91      0.87      0.89      1000
weighted avg       0.95      0.95      0.95      1000


  XGBoost
              precision    recall  f1-score   support

           0       0.96      0.98      0.97       850
           1       0.85      0.77      0.80       150

    accuracy                           0.94      1000
   macro avg       0.90      0.87      0.89      1000


## 24 · Error Analysis

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]
errors = y_test != best_pred
error_rate = errors.mean()
print(f"Best model: {best_name}")
print(f"Error rate: {error_rate:.4f} ({errors.sum()} / {len(y_test)})")
print(f"\nErrors by true class:")
error_df = pd.DataFrame({'true': y_test, 'pred': best_pred, 'error': errors})
print(error_df.groupby('true')['error'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'errors', 'count': 'total', 'mean': 'error_rate'}))

Best model: CatBoost
Error rate: 0.0510 (51 / 1000)

Errors by true class:
      errors  total  error_rate
true                           
0         15    850    0.017647
1         36    150    0.240000


## 25 · Interpretation and Business Insight

- **Claim amount** is the top predictor — fraud claims are significantly higher.
- **Incident severity** (total_loss, severe) correlates strongly with fraud.
- **Fewer witnesses** and **no police report** are fraud red flags.
- **Prior claims** history is a strong indicator of serial fraud.
- **Newer customers** (low months_as_customer) have higher fraud rates.

## 26 · Limitations

1. Synthetic data — real insurance fraud is more subtle.
2. Only 14 features — real systems use 100+ including text from claim descriptions.
3. No claim text or adjuster notes.
4. No network features (related claimants, providers).
5. Static analysis — no temporal fraud ring detection.

## 27 · How to Improve This Project

1. Add NLP features from claim descriptions and adjuster notes.
2. Engineer claim-to-premium ratio and claim frequency features.
3. Build network graphs to detect fraud rings.
4. Apply cost-sensitive learning (fraud cost vs investigation cost).
5. Use SHAP for explainable fraud scores for investigators.

## 28 · Production Considerations

- Score claims at submission for triage prioritization.
- Human investigators review high-score claims.
- Track investigation outcomes for model retraining.
- Regulatory requirements for decision explainability.
- SIU (Special Investigations Unit) workflow integration.

## 29 · Common Mistakes

1. Using accuracy when fraud is 15% of claims.
2. Not considering investigation capacity constraints.
3. Ignoring the cost of false positives (investigating legitimate claims).
4. Not tracking model degradation over time.
5. Deploying without investigator feedback loop.

## 30 · Mini Challenge / Exercises

1. Calculate the cost savings if you catch 80% of fraud with 20% false positive rate.
2. Add a claim_to_premium_ratio feature and evaluate impact.
3. Compare model performance with and without prior_claims feature.
4. Build a simple rule-based system and compare to ML.

## 31 · Final Summary / Key Takeaways

1. Insurance fraud detection combines ML with domain expertise.
2. Claim amount, severity, and prior claims are key fraud indicators.
3. Fewer witnesses and no police report are behavioral red flags.
4. Explainability is essential — investigators need reasons.
5. Production systems need cost-sensitive thresholds and human review.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average='weighted')), 4),
        "precision": round(float(precision_score(y_test, yp, average='weighted')), 4),
        "recall": round(float(recall_score(y_test, yp, average='weighted')), 4),
    }
with open('metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Metrics saved.")
print(json.dumps(metrics_out, indent=2))

Metrics saved.
{
  "CatBoost": {
    "accuracy": 0.949,
    "f1": 0.9474,
    "precision": 0.9474,
    "recall": 0.949
  },
  "LightGBM": {
    "accuracy": 0.937,
    "f1": 0.9342,
    "precision": 0.9344,
    "recall": 0.937
  },
  "XGBoost": {
    "accuracy": 0.944,
    "f1": 0.9429,
    "precision": 0.9424,
    "recall": 0.944
  },
  "Logistic Regression": {
    "accuracy": 0.909,
    "f1": 0.9061,
    "precision": 0.9047,
    "recall": 0.909
  },
  "FLAML": {
    "accuracy": 0.947,
    "f1": 0.9455,
    "precision": 0.9453,
    "recall": 0.947
  }
}
